In [ ]:


# ==============================
# 🚀 Instalación de librerías
# ==============================
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q accelerate
!pip install -q faiss-cpu
!pip install -q ipywidgets

# ==============================
# 🔹 Importación de librerías
# ==============================
import torch
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import threading
import time

# ==============================
# 🔹 Verificar GPU
# ==============================
print("GPU disponible:", torch.cuda.is_available())

# ==============================
# 🔹 Documentos de ejemplo
# ==============================
documents = [
    "Python es un lenguaje de programación muy usado en inteligencia artificial.",
    "FAISS es una biblioteca creada por Facebook para búsqueda vectorial eficiente.",
    "Los embeddings convierten texto en vectores numéricos.",
    "RAG significa Retrieval Augmented Generation.",
    "LangChain permite construir aplicaciones con modelos de lenguaje.",
    "Kubernetes es una plataforma para orquestación de contenedores.",
    "Docker permite crear y ejecutar aplicaciones en contenedores.",
    "Linux es un sistema operativo basado en Unix."
]

# ==============================
# 🔹 Dividir documentos en fragmentos
# ==============================
text_splitter = CharacterTextSplitter(chunk_size=150, chunk_overlap=20)
docs = text_splitter.create_documents(documents)

# ==============================
# 🔹 Crear embeddings y vectorstore FAISS
# ==============================
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

# ==============================
# 🔹 Cargar modelo de lenguaje
# ==============================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ==============================
# 🔹 Función para generar respuesta del modelo
# ==============================
def generate_response(prompt, max_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

# ==============================
# 🔹 Configuración de la interfaz RAG
# ==============================
# Área de texto
text_input = widgets.Textarea(
    placeholder='Escribe tu pregunta aquí...',
    layout={'width': '100%', 'height': '100px'}
)

# Botón
button = widgets.Button(
    description='Preguntar',
    button_style='success',
    icon='search'
)

# Área de salida
output_area = widgets.Output(
    layout={'width': '100%', 'max_height': '400px', 'overflow_y': 'auto'}
)

# ==============================
# 🔹 Animación de carga
# ==============================
loading = False

def loading_animation():
    dots = ""
    while loading:
        dots += "."
        if len(dots) > 3:
            dots = ""
        with output_area:
            clear_output(wait=True)
            display(Markdown(f"### ⏳ Generando respuesta{dots}"))
        time.sleep(0.5)

# ==============================
# 🔹 Función RAG
# ==============================
def run_rag(query):
    # Buscar los 2 documentos más similares
    results = vectorstore.similarity_search(query, k=2)
    context = "\n".join([doc.page_content for doc in results])

    # Crear prompt para el modelo
    prompt = (
        "Responde únicamente usando el contexto proporcionado.\n"
        "No hagas preguntas adicionales.\n"
        "No agregues información externa.\n"
        "Responde en máximo 3 líneas.\n\n"
        f"Contexto:\n{context}\n\n"
        f"Pregunta: {query}\n\n"
        "Respuesta:"
    )

    response = generate_response(prompt)

    # Limpieza de respuesta
    if "Pregunta:" in response:
        response = response.split("Pregunta:")[0].strip()

    return context, response

# ==============================
# 🔹 Función para el botón
# ==============================
def on_button_clicked(b):
    global loading
    query = text_input.value.strip()
    if not query:
        return

    loading = True
    thread = threading.Thread(target=loading_animation)
    thread.start()

    # Ejecutar RAG
    context, response = run_rag(query)

    loading = False
    thread.join()

    with output_area:
        clear_output(wait=True)
        display(Markdown("### 📚 Contexto Recuperado"))
        display(Markdown(context))
        display(Markdown("---"))
        display(Markdown("### 💬 Respuesta"))
        display(Markdown(response))

# ==============================
# 🔹 Conectar botón a función
# ==============================
button.on_click(on_button_clicked)

# ==============================
# 🔹 Mostrar interfaz
# ==============================
ui = widgets.VBox([text_input, button, output_area])
display(ui)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.1 MB/s eta 0:00:00
GPU disponible: False


/tmp/ipykernel_913/3263815314.py:54: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]